# 00 — Theory & Statistics Reference

> This is the **first notebook in the handbook**, and the one to keep open in a
> side tab while you read any other. It is *not* a tutorial and *not* a library
> API reference — it is the place you come back to when a notebook uses a term
> and you want the precise definition, the formula, and **why it matters**.
>
> Other notebooks link here as ` §3.2` (section 3, sub 2). Jump straight there.

## Why this notebook exists

The same word means different things in probability theory, classical
statistics, machine learning, and business. If you don't know which meaning a
notebook is using, you'll quietly misunderstand the conclusion. This reference
disambiguates exactly those traps. We start with the master table — the four
"dictionary collisions" that cause the most real-world damage — then go deeper
into each concept below.

## §0 — The Data Science Dictionary Collisions

> Read this table once, return to it often. Each row is a word that has **three
> or four legitimate meanings**, and the "trap" column is where the money is:
> the specific mistake people make when they confuse them.

| Term | The Statistical / Math Meaning | The Machine Learning / CS Meaning | The Business / Colloquial Meaning | The Trap |
|---|---|---|---|---|
| **Bias** | **Estimator error:** the difference between an estimator's expected value and the true parameter value, $E[\hat\theta]-\theta$. | **An intercept:** the constant $b$ added to a weighted sum in a neural network or regression ($y = wx + b$). | **Prejudice:** unfair discrimination against certain groups in algorithmic outcomes. | A model can have **zero** statistical bias, a **heavily tuned** mathematical bias weight, and still exhibit **severe** societal bias. |
| **Variance** | **Data spread:** the expected value of the squared deviation from the mean, $\sigma^2$. | **Model sensitivity:** how much a model's predictions fluctuate if trained on a different subset of data. | *N/A* | Confusing the natural variance **of the data** with the variance **of the model** (as in the bias–variance trade‑off). |
| **Significance** | **Statistical ($p$‑value):** the probability of observing the data given that the null hypothesis is true. | *N/A* | **Importance:** a large, practically meaningful impact or magnitude. | A result can be **statistically significant** (e.g. a 0.001% lift in conversion) yet **entirely practically insignificant** to the business. |
| **Independent** | **Variables:** the inputs/predictors ($X$) used to estimate the dependent variable ($Y$). | **Linear algebra:** vectors that cannot be written as a combination of each other (crucial for matrix operations). | **Probability:** two events where the outcome of one does not affect the other, $P(A \cap B) = P(A)P(B)$. | Assuming your "independent variables" (features) in a dataset are actually **statistically independent** of one another — they rarely are. |

**The single habit that prevents most analytical misunderstandings:** when a
notebook or colleague uses any word from this table, mentally ask *which
meaning is in play?* The detailed sections below expand each row.

### How to use the rest of this reference

- Each section is self‑contained — jump straight to the one you need.
- **Formulas** appear both in math and in plain words.
- **"Why it matters"** explains when a working data scientist touches this.
- **"Watch out"** flags the common misreadings.
- Cross‑references like `(see §3.2)` point to related sections.

---
# §1. Describing a single variable

## §1.1 Measures of centre (mean, median, mode)

| Measure | Formula | Plain words | When |
|---|---|---|---|
| **Mean** $\bar{x} = \frac{1}{n}\sum x_i$ | add up, divide by count | the "average" | symmetric data, no extreme outliers |
| **Median** | middle value when sorted | half the data below, half above | skewed data, data with outliers (income, prices) |
| **Mode** | most frequent value | the value that appears most | categorical data, or discrete counts |

**Why it matters:** on right‑skewed data (salaries, house prices, web traffic)
the mean sits **above** the median because a few huge values drag it up. Quoting
"average income" vs "median income" can move a policy argument by 30%+. The
same is true for "average order value" in e‑commerce.

**Watch out:** the mean has no resistance to outliers — one billionaire in a
sample of 100 makes the mean misleading. When a notebook reports a mean on
skewed data, check the median too (and the histogram).

## §1.2 Measures of spread (variance, std, IQR, range)

| Measure | Formula | Plain words |
|---|---|---|
| **Variance** $\sigma^2 = \frac{1}{n}\sum(x_i-\bar{x})^2$ | average squared distance from the mean |
| **Standard deviation (std)** $\sigma = \sqrt{\sigma^2}$ | spread in the **same units** as the data |
| **Range** | max − min | total extent; very outlier‑sensitive |
| **IQR** (interquartile range) | Q3 − Q1 | spread of the middle 50%; outlier‑resistant |

> **Why square the deviations in variance?** Raw deviations sum to zero
> (positive and negative cancel). Squaring makes everything positive and
> penalises large deviations more — usually what we want.

**Why it matters (two big reasons):**
- Distance‑ and gradient‑based models (k‑NN, SVM, neural nets) compare features
 by *magnitude*. If one feature has std 1000 and another 0.1, the model treats
 the big one as vastly more important. **Standardising** (dividing by std)
 fixes this.
- Variance is the "noise floor" your model is trying to beat (see §6.1).

**Watch out — variance is overloaded** (see §0 above and §6.1 below): data
spread, model sensitivity, and the bagging‑reducible quantity are *three
related but distinct ideas*. Always ask which one a notebook means.

## §1.3 Shape: skewness and kurtosis

- **Skewness** measures asymmetry. Right‑skew (long tail right, mean > median)
 is the common case for money/count data. Left‑skew is its mirror.
- **Kurtosis** measures tail heaviness ("how often do extreme values happen?").
 High kurtosis = fat tails = more black‑swan events than a normal distribution
 would predict.

**Why it matters:** linear and distance‑based models behave better on roughly
symmetric features. A **log transform** tames right‑skew and is the single most
useful reshaping trick in practice (see notebook 03, EDA & feature engineering).

**Watch out:** "I log‑transformed it" is not magic. Check skewness before and
after; if the histogram still has a heavy tail, consider a Box‑Cox or quantile
transform instead.

## §1.4 Distributions you'll meet

| Distribution | Shape | Models |
|---|---|---|
| **Normal (Gaussian)** | bell curve | heights, measurement errors, residuals of a good linear model |
| **Uniform** | flat | random assignment (A/B group allocation) |
| **Bernoulli** | 0 or 1 | a single yes/no outcome (click / no click) |
| **Binomial** | counts of successes in n trials | conversions out of n visitors |
| **Poisson** | counts of rare events | arrivals per hour, defects per batch |
| **Exponential** | decay | time until next event |
| **Log‑normal** | right‑skewed | incomes, prices, web session lengths |

**Why it matters:** many tests and models *assume* a distribution. The t‑test
assumes roughly normal residuals; Poisson regression assumes count outcomes. If
your data grossly violates the assumption, the test/model is unreliable. Always
plot first.

---
# §2. Describing relationships between two variables

## §2.1 Covariance and correlation

**Covariance** measures how two variables move together:

$$\mathrm{cov}(X, Y) = \frac{1}{n}\sum_i (x_i-\bar{x})(y_i-\bar{y})$$

- Positive → they tend to go up together.
- Negative → one goes up while the other goes down.
- Near zero → no *linear* relationship.

Covariance is hard to read because its units are "units of X × units of Y".
**Pearson correlation** normalises it to a fixed [−1, +1] range:

$$r = \frac{\mathrm{cov}(X, Y)}{\sigma_X \, \sigma_Y} \in [-1, 1]$$

| r value | interpretation |
|---|---|
| +1 | perfect positive linear relationship |
| +0.7 to +0.9 | strong positive |
| +0.3 to +0.7 | moderate positive |
| 0 to ±0.3 | weak / negligible |
| −1 | perfect negative linear relationship |

**Why it matters:**
- A correlation heatmap is the fastest way to find candidate predictors and
 redundant features (see notebook 03).
- Highly correlated *features* (|r| > 0.8–0.9) are often redundant — keeping
 both destabilises linear‑model coefficients (multicollinearity, see §7.2).

**Watch out (two big ones):**
1. **Correlation only captures *linear* relationships.** A perfect U‑shaped
 relationship has r ≈ 0. Always plot the scatterplot, not just the number.
2. **Correlation ≠ causation.** See §3 immediately below — this is the whole
 reason this reference exists.

---
# §3. Correlation vs Causation (the disambiguation that motivated this notebook)

The single most misunderstood concept in applied analytics. The word
"correlated" has two meanings, and conflating them causes real‑world harm.

## §3.1 The two meanings of "correlated"

| In **probability / statistics** | In **data science / analytics** |
|---|---|
| Two random variables whose covariance is non‑zero. A purely mathematical property of their joint distribution. | A pattern in data that we hope reflects a real‑world driver — i.e. that changing one *will* change the other. |

In probability theory, correlation is just a number. It says nothing about
*why*. Two variables can be correlated because:

- **A causes B** (the thing you usually want),
- **B causes A** (reverse causation),
- a **third variable C causes both** (confounding — the dangerous one),
- or pure **coincidence** (spurious correlation).

The classic example: "ice cream sales and shark attacks" rise together — both
driven by **summer weather**, not by each other.

## §3.2 Confounding: the silent prediction‑killer

A **confounder** is a third variable that influences *both* the predictor and
the outcome. If you don't account for it, your model learns the confounder's
shadow instead of the real signal — and **its predictions break the moment the
confounder changes**.

> **Definition (formal).** Variable $Z$ is a confounder of the $X \to Y$
> relationship if $Z$ affects both $X$ and $Y$, and $Z$ is not on the causal
> path from $X$ to $Y$.

**Why it destroys predictions:** your model fit on historical data where "ad
spend" and "sales" rose together because the economy was booming. The model
credits the ads. When the economy turns and you keep spending on ads, sales
*fall* — the model never learned about the economy, so it can't adjust. The
correlation was real; the causation was an illusion. (Notebook 04, Regression,
turns this into a worked Simpson's‑paradox example you can run.)

## §3.3 Simpson's paradox: confounding made visible

Simpson's paradox is what happens when ignoring a confounder **flips the sign**
of your conclusion. A trend that holds in every subgroup **reverses** when you
pool the groups.

**The textbook case (UC Berkeley admissions, 1973):** overall, men were
admitted at a higher rate than women — *looks like bias against women*. But
department‑by‑department, women were admitted at *equal or higher* rates. The
confounder was **department choice**: women applied to more competitive
departments. Once you condition on department, the apparent bias disappears (and
in some readings reverses). Pooling hid the truth.

> Simpson's paradox is just **omitted variable bias in its most dramatic form**
> — see §7.3. The fix is always the same: identify the confounder and condition
> on it (include it in the model).

## §3.4 So how *do* you prove causation?

You cannot prove causation from observational data alone — you can only argue
for it. Strength of evidence, increasing:

| Method | Strength | Idea |
|---|---|---|
| **Randomised controlled experiment (A/B test)** | gold standard | random assignment breaks confounding by distributing confounders equally across arms (see notebook 05) |
| **Natural experiment / instrumental variable** | strong | find a "random nudge" that affects X but not Y except through X |
| **Difference‑in‑differences** | moderate | compare trends in a treatment group vs a similar control group |
| **Causal graph + adjustment** (do‑calculus, back‑door criterion) | strong *if your graph is right* | explicitly list confounders and adjust for them |
| **Granger causality** | weak (not really causal) | "X helps predict Y's future" — but prediction isn't causation |
| **Plain correlation** | not causal at all | a starting hint only |

## §3.5 The two causal frameworks (vocabulary you'll see)

If you read further into causality, you'll meet two largely equivalent
formalisms:

| Framework | Core idea | Vocabulary |
|---|---|---|
| **Potential outcomes** (Neyman / Rubin) | Causality is defined by the **counterfactual**: what *would have* happened to the *same* unit under the other treatment. | potential outcomes $Y(1), Y(0)$; average treatment effect $E[Y(1)-Y(0)]$ |
| **Structural causal models / do‑calculus** (Pearl) | Causality is encoded in a **directed acyclic graph (DAG)** of variables; interventions are $P(Y \mid \mathrm{do}(X))$. | DAG, back‑door criterion, confounders, colliders |

For *interventional* questions ("what if I set X?") the two frameworks give the
same answers. They diverge on deeper counterfactual questions, beyond this
handbook. The practical takeaway: **a randomised experiment sidesteps the whole
debate** — random assignment makes the confounders balance out by construction.

## §3.6 Practical checklist before you claim "X drives Y"

1. **Plot X vs Y, and also vs every plausible confounder.**
2. **Ask: is there a plausible third variable that affects both?** If yes,
 measure it and condition on it.
3. **Check stability across subgroups** (the Simpson check). Does the effect
 survive when you split by segment, time period, geography?
4. **Ask: could the arrow point the other way?** (reverse causation).
5. **If the decision is high‑stakes, run an experiment** (notebook 05) — it's
 the only way to settle it cleanly.

---
# §4. Probability essentials

## §4.1 Joint, marginal, conditional probability

For two events $A$, $B$:

- **Joint** $P(A, B)$ — prob both happen.
- **Marginal** $P(A) = \sum_B P(A, B)$ — prob of $A$ regardless of $B$.
- **Conditional** $P(A \mid B) = \frac{P(A, B)}{P(B)}$ — prob of $A$ given
 that $B$ happened.

**Bayes' theorem** flips the conditioning:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}$$

**Why it matters:** logistic regression outputs $P(\text{class} \mid
\text{features})$. AUC evaluates the *ranking* implied by those probabilities
(notebook 04). Naive Bayes (a classifier) is literally Bayes' theorem with a
strong independence assumption.

**Watch out — the base‑rate fallacy:** a test that is "99% accurate" for a rare
disease (prevalence 1 in 10,000) still produces mostly false positives, because
the base rate is so low. Always fold in $P(A)$ (the prior).

## §4.2 Independence vs uncorrelatedness

| Term | Meaning |
|---|---|
| **Independent** | knowing X tells you *nothing* about Y: $P(Y \mid X) = P(Y)$ |
| **Uncorrelated** | the *linear* relationship is zero: $\mathrm{cov}(X, Y) = 0$ |

**Independence implies uncorrelatedness, but NOT vice versa.** $Y = X^2$ with
$X$ centred on 0 is uncorrelated with $X$ (covariance is 0) but obviously *not*
independent — knowing $X$ tells you $Y$ exactly.

**Why it matters:** the Naive Bayes classifier *assumes* features are
conditionally independent given the class — almost never literally true, but
often "true enough" to work well. The distinction also explains why a zero on a
correlation heatmap does **not** mean "no relationship" (see §2.1). This is the
"independent" trap from §0 in detail.

## §4.3 Bayes' theorem

Bayes' theorem flips a conditional — it turns a probability you *can* measure into
the one you *actually want*:

$$\boxed{\;P(H \mid D) = \frac{P(D \mid H)\, P(H)}{P(D)}\;}$$

| Symbol | Name | Meaning |
|---|---|---|
| $P(H)$ | **prior** | how likely the hypothesis was *before* the data |
| $P(D \mid H)$ | **likelihood** | how likely the data is *if* $H$ is true |
| $P(H \mid D)$ | **posterior** | how likely $H$ is *after* seeing the data (what we want) |
| $P(D)$ | **evidence** | normaliser: $\sum_H P(D \mid H)P(H)$ |

**Why it matters:** in disease screening, fraud detection, and spam filtering you
measure $P(\text{test+} \mid \text{sick})$ (the likelihood / test accuracy) but you
*want* $P(\text{sick} \mid \text{test+})$ (the posterior). Bayes does the flip —
and the flip is non-intuitive: a 99%-accurate test for a 1%-prevalence disease
yields a posterior of only ~50%, because the false positives from the large healthy
population dominate the denominator. This is **base-rate neglect**, the most common
probability error. Naive Bayes classifiers, logistic regression's interpretation,
and the precision/recall trade-off (low prevalence kills precision) all reduce to
this one equation. See notebook 04½ §4 for the worked example and prevalence sweep.

## §4.4 Bayesian vs frequentist inference

Two coherent philosophies of what "probability" *means*, with practical
consequences (§0):

| | **Frequentist** | **Bayesian** |
|---|---|---|
| Probability is… | long-run frequency | degree of belief |
| Parameters are… | fixed unknown constants | random, with distributions |
| Output | point estimate + confidence interval | full posterior distribution |
| "Prior" | not used (or: implicit) | explicit, must be chosen |
| Typical tools | t-test, ANOVA, MLE, p-values (notebook 05) | Bayes' theorem, posteriors, credible intervals, Naive Bayes (notebook 04½) |

**Why it matters:** most classical statistics in this handbook (hypothesis tests,
notebook 05; linear regression, notebook 04) is frequentist. Most *ML* (softmax
probabilities, logistic regression's $P(y=1)$, regularisation-as-a-prior, VAEs) is
quietly Bayesian. The two frameworks give similar answers with lots of data and
weak priors, but diverge with little data or strong prior beliefs. L2 regularisation
(notebooks 04, 15) is mathematically equivalent to placing a Gaussian prior on the
weights — a Bayesian reading of a frequentist technique.

---
# §5. Hypothesis testing & uncertainty

This is the bedrock of notebook 05 (Validation, Hypothesis Testing & A/B
Tests). The bare essentials:

## §5.1 The five parts of every test

| Part | What it is |
|---|---|
| **Null hypothesis $H_0$** | "nothing interesting" — e.g. means are equal, no effect |
| **Alternative $H_1$** | the claim you're hunting for — e.g. treatment > control |
| **Test statistic** | a number summarising how far the data is from $H_0$ |
| **p‑value** | if $H_0$ were true, how often would we see a statistic this extreme? |
| **Decision** | reject $H_0$ if p < $\alpha$ (your false‑positive tolerance, usually 0.05) |

## §5.2 The four numbers (memorise these)

| Concept | Meaning | Intuition |
|---|---|---|
| **$\alpha$ (significance)** | false‑positive rate you accept | "cry wolf" tolerance |
| **$\beta$ (type II error)** | false‑negative rate | miss rate |
| **Power $= 1 - \beta$** | prob of detecting a true effect | sensitivity; target ≥ 0.8 |
| **Effect size** | how big the effect actually is | what you actually care about |

## §5.3 The two ways to be wrong

| | $H_0$ is actually true | $H_0$ is actually false |
|---|---|---|
| **Fail to reject $H_0$** | correct | **Type II** (false negative — missed it) |
| **Reject $H_0$** | **Type I** (false positive — cried wolf) | correct |

**Why it matters:** $\alpha$ controls Type I; sample size + effect size control
Type II (via power). Notebook 05 computes power *before* collecting data so you
don't run an experiment that can't detect anything.

**Watch out:** "p = 0.06" does **not** mean "probably no effect". It means
"insufficient evidence to rule out chance". Absence of evidence ≠ evidence of
absence. This is the "significance" trap from §0 in detail.

---
# §6. How models learn: bias, variance, and the trade‑off

This section is the heart of notebook 02 (Introduction to ML).

## §6.1 The bias–variance decomposition

For a prediction $\hat{y}$ of a true value $y = f(x) + \varepsilon$ (where
$\varepsilon$ is irreducible noise), the expected squared error breaks into
three parts:

$$\underbrace{E[(y - \hat{y})^2]}_{\text{total error}} = \underbrace{(E[\hat{y}] - f(x))^2}_{\text{bias}^2} + \underbrace{E[(\hat{y} - E[\hat{y}])^2]}_{\text{variance}} + \underbrace{\sigma^2_\varepsilon}_{\text{irreducible}}$$

- **Bias** — how wrong the model is *on average*. High bias = underfitting (the
 model is too simple; a linear model trying to fit a parabola).
- **Variance** — how much the model's predictions wobble *between training
 sets*. High variance = overfitting (it memorises each training set).
- **Irreducible error** — the noise floor. No model can beat this.

## §6.2 The trade‑off

| Dial it up | Bias | Variance | Example |
|---|---|---|---|
| More model complexity (deeper tree, higher‑degree polynomial) | ↓ | ↑ | overfit risk |
| Less model complexity (shallow tree, linear) | ↑ | ↓ | underfit risk |
| More training data | ~same | ↓ | almost always helps |
| Regularisation (Ridge/Lasso, max_depth, dropout) | ↑ | ↓ | the main overfit cure |

**Why it matters:** every modelling choice is a bias–variance trade‑off.
Notebook 02 visualises it with polynomials of increasing degree. The cure for
overfitting is **more data**, **regularisation**, or **a simpler model**; for
underfitting it's **a more flexible model** or **better features**.

**Watch out:** this "bias" and "variance" are the ML/CS meanings from §0 — model
error and model sensitivity — *not* the data‑spread variance of §1.2. Same
words, related ideas, distinct quantities. Random forests (notebook 08) exploit
this by *averaging* high‑variance trees to drop the variance term.

---
# §7. Regression‑specific concepts

## §7.1 $R^2$ — fraction of variance explained

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- 1 = perfect predictions; 0 = no better than guessing the mean; negative =
 *worse* than guessing the mean (only possible on test data, signals a broken
 model).
- It's a proportion, not an error — higher is better.

**Why it matters:** it's the default regression metric and a quick health
check. But it's only meaningful on data the model didn't train on.

**Watch out:**
- Adding *any* feature to a linear model **never decreases training $R^2$**,
 even garbage features. So training $R^2$ is a lie about generalisation.
- $R^2$ doesn't tell you the *error magnitude*. A model can have $R^2 = 0.99$
 and still be off by an unacceptable amount in absolute terms. Always pair it
 with RMSE/MAE in the target's units.

## §7.2 Multicollinearity

When two or more predictors are highly correlated with each other, the model
can't tell which one deserves the credit. Consequences:

- Individual coefficients become **unstable** (swing wildly with small data
 changes, sometimes flipping sign).
- Their standard errors blow up, so p‑values become unreliable.
- The **overall** prediction is usually fine; only the *interpretation* of
 individual coefficients suffers.

**Why it matters:** if you're reading a regression coefficient to make a
business claim ("each extra dollar of security investment reduces churn by
X"), multicollinearity can give you a confidently wrong number. Diagnose with a
correlation heatmap or VIF (variance inflation factor); fix by dropping one of
the redundant features, combining them, or using Ridge regression.

> **Connection to confounding (§3):** multicollinearity is about *predictors*
> being correlated with each other; confounding is about a *predictor* being
> correlated with the *outcome via a hidden third variable*. Both destabilise
> coefficient interpretation, but for different reasons.

## §7.3 Omitted variable bias (OVB)

If a variable that (a) affects the outcome and (b) is correlated with an
included predictor is **left out** of the model, the coefficients on the
included predictors become **biased** — they absorb the effect of the missing
variable.

$$\hat{\beta}_{\text{included}} = \beta_{\text{true}} + \underbrace{\delta \cdot \gamma}_{\text{OVB}}$$

where $\delta$ is the effect of the omitted variable on the outcome and
$\gamma$ is its relationship with the included predictor.

**Why it matters:** this is the regression‑flavoured version of confounding
(§3.2) and the engine behind Simpson's paradox (§3.3). Your coefficient can be
the wrong sign, the wrong magnitude, or both — purely because you didn't
measure something.

**The hard truth:** every observational regression has *some* OVB, because you
can never measure everything. The question is whether the bias is small enough
to live with. Randomised experiments (notebook 05) eliminate OVB entirely by
construction.

---
# §8. Glossary of other overloaded / confusing terms

The four worst offenders are in §0. Here are the rest.

| Term | Meaning A (probability / classical stats) | Meaning B (ML / data science) | Notes |
|---|---|---|---|
| **correlation** | non‑zero covariance (purely mathematical) | a hoped‑for real‑world driver (causal) | §3 |
| **error** | residual = $y - \hat{y}$ | test‑set prediction mistake | context decides |
| **error rate** | misclassification rate, $1 - \text{accuracy}$ | sometimes informally = RMSE | don't mix |
| **feature** | (stats) "explanatory variable" / "covariate" | (ML) a model input column | synonyms |
| **regularisation** | adding a penalty to the loss to shrink coefficients | any technique that combats overfitting (dropout, pruning) | §6.2 |
| **sample** | a single draw / row of data | the whole dataset ("a sample of customers") | ambiguous |
| **significant figures** | (sciences) decimal precision | nothing to do with statistical significance | only coincidentally similar spelling |

> **The rule:** when a notebook or a colleague says any word from §0 or this
> table, mentally ask "which meaning?" That single habit prevents most
> analytical misunderstandings.

---

# §9. Deep‑learning glossary (notebooks 16–18)

These terms appear in the Phase 2 (Deep Learning) notebooks. They are defined here
once so each notebook can point at ` §9.x` instead of re‑explaining.

## §9.1 Receptive field

In a CNN, a neuron's **receptive field** is the region of the *input* image that can
influence its output. A single `3×3` conv unit sees a `3×3` patch — a tiny receptive
field. But after stacking conv layers, deeper neurons see *combinations* of earlier
patches, so their effective receptive field grows: layer‑2 neurons each see a `5×5`
region of the original image (a `3×3` window of `3×3` windows). This is *why* "early
layers detect edges, deeper layers detect objects" — it's a direct consequence of
the receptive field growing with depth. (Notebook 16 §7 visualises this with feature
maps.)

## §9.2 Parameter sharing & local connectivity

A fully‑connected (dense) layer gives every input its own weight. A conv layer
**shares** one small filter across the entire input — the same `3×3` weight matrix
slides over every position. Two consequences (both desirable for images):

- **Parameter efficiency** — a `3×3` filter has 9 weights regardless of image size,
 vs. an MLP's one‑weight‑per‑pixel. This is why CNNs scale to large images
 (notebook 16 §1 shows the parameter‑explosion curve an MLP suffers).
- **Translation equivariance** — if a pattern (an edge, a cat) appears at a different
 location, the same filter still detects it. The MLP loses this by flattening.

## §9.3 Skip / residual connections (ResNet)

A **skip connection** adds a layer's *input* to its *output*: $y = F(x) + x$. The
network learns the *residual* $F(x)$, not the full mapping. Two effects: (1) the
gradient can flow straight back through the `+x`, so very deep networks (50–152
layers) become trainable instead of suffering vanishing gradients; (2) if a layer
isn't helping, it can learn $F \approx 0$ and pass `x` through unchanged. This is
the same "give the gradient a clean additive path" idea that powers the LSTM's cell
state (§9.4). Notebook 16 §8 covers the ResNet lineage.

## §9.4 Backpropagation through time (BPTT) & vanishing/exploding gradients

An RNN unrolled over `T` timesteps is just a deep feed‑forward net of depth `T` with
shared weights. Training it = backprop on that unrolled graph = **BPTT**. The
gradient at early timesteps passes through a product of `T` Jacobians:

$$\frac{\partial L}{\partial h_0} \;\propto\; \prod_{t=1}^{T} W_h \cdot \sigma'(\cdot)$$

- If that product's magnitude is `< 1`, repeated `T` times it shrinks exponentially
 → **vanishing gradient**: early timesteps get no learning signal, long‑range
 dependencies can't be learned.
- If `> 1`, it grows → **exploding gradient**: weights blow up to NaN. (Cure:
 gradient clipping.)
- The factor $\sigma'(\cdot)$ makes it worse: `tanh'` ≤ 1 (often ~0.5), and `sigmoid'`
 ≤ 0.25 (see notebook 15 §1.3 — the same vanishing mechanism, here multiplied per
 timestep).

The **LSTM** fixes vanishing gradients by making the cell‑state update *additive*
($C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$): the gradient through the `+`
is ~1 per step, so it survives. (Same principle as the skip connection, §9.3.)
Notebook 17 §3 demonstrates the vanishing gradient on the adding task, and §4 derives
the LSTM gates.

## §9.5 Embeddings

An **embedding** is a learned lookup table mapping a discrete ID (a word, a product,
a category) to a dense vector of `d` numbers. The table's weights are trained by
backprop like any other. After training, IDs used in similar contexts end up with
similar vectors — the network discovers a notion of "meaning" from co‑occurrence.

For text, the embedding turns a sequence of word IDs into a `(seq_len × d)` matrix
the RNN/CNN/transformer can process. For tabular categorical features, embeddings
often beat one‑hot for high‑cardinality columns (fewer parameters, and the learned
vector captures similarity). The cardinality × `d` params in the table are usually
the bulk of a small model's parameter count — see notebook 17 §5.

## §9.6 Bottleneck, representation learning & the autoencoder–PCA link

A **bottleneck** is a layer narrower than its input/output. Forcing a network's
information through a bottleneck compels it to learn a compressed **representation**
(code) that retains only the structure needed to reconstruct the input — this is
**representation learning**, learning useful features without labels.

The **autoencoder** (notebook 18) is the canonical example: `encoder` (input →
code) + `decoder` (code → reconstruction), trained with reconstruction loss
$\|x - \hat{x}\|^2$. Two facts worth pinning down:

- A **linear** autoencoder (no activations, MSE loss) recovers the **PCA** subspace
 (notebook 11) — same principal components, up to rotation. PCA *is* the linear
 limit of an autoencoder.
- A **nonlinear** autoencoder (ReLU activations) can in principle beat PCA by
 learning a curved (manifold) compression. In practice the margin is often small
 for pure reconstruction, and the autoencoder earns its keep on **denoising**,
 **anomaly detection** (high reconstruction error = outlier), and as a **feature
 extractor** for downstream tasks — none of which PCA was designed for.

The bottleneck's width (`code_dim`) is the central hyperparameter: too narrow and
information is lost; too wide and the net trivially copies (the identity) unless an
extra constraint (denoising / sparsity / contraction) is added. See notebook 18 §3
for the code-size trade-off curve.

---
# §10. Where to go deeper

Each entry is a one‑click jump into the canonical source.

- **Causality** — Pearl, *Causality* (2009); Cunningham, *The Mixtape*
 <https://mixtape.scunning.com/>; the inference.vc intro
 <https://www.inference.vc/untitled/>.
- **Simpson's paradox** — <https://en.wikipedia.org/wiki/Simpson%27s_paradox>;
 Statistics by Jim <https://statisticsbyjim.com/basics/simpsons-paradox/>.
- **Hypothesis testing & power** — Wasserman, *All of Statistics*, Ch. 8–11.
- **Bias–variance** — scikit‑learn's underfitting/overfitting doc
 <https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html>.
- **Experiments / A/B testing** — Kohavi et al., *Trustworthy Online Controlled
 Experiments* (2020).
- **Deep learning** — Goodfellow, Bengio & Courville, *Deep Learning* (2016),
 <https://www.deeplearningbook.org/>; the course lectures on CNNs/RNNs/autoencoders.